### Sampling Places

![German renewable-weather sampling points](german_sampling_points_map.png)

The sampling design is based on real German renewable generation geography:

- Wind generation is concentrated in northern Germany and offshore zones.
- Solar PV is strongest in Bavaria, Baden-Wuerttemberg, North Rhine-Westphalia,
  and other high-installation states.
- We only sample for wind and solar generation.
- Curtailment can be driven by different spatial patterns: northern wind
  congestion, offshore wind, or southern/distribution-grid solar pressure.

We then group these sampling points by their geographical position and main type of generation.

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RAW_DATA_PATH = Path("../raw_data")
PROCESSED_DATA_PATH = Path("../processed_data")
START_DATE = "2019-05-01"
END_DATE = "2026-05-01"

OBSERVED_COMBINED_PATH = RAW_DATA_PATH / f"Open_Meteo_combined_{START_DATE}_to_{END_DATE}.csv"
REGIONAL_FEATURES_PATH = PROCESSED_DATA_PATH / f"Open_Meteo_regional_features_{START_DATE}_to_{END_DATE}.csv"

regional_features = pd.read_csv(REGIONAL_FEATURES_PATH, parse_dates=["timestamp_utc"])
regional_features["timestamp_utc"] = pd.to_datetime(regional_features["timestamp_utc"], utc=True)

# Recreate local-time columns from UTC so the notebook does not depend on how
# timezone-aware timestamps were serialized in the processed CSV.
regional_features["timestamp_de"] = regional_features["timestamp_utc"].dt.tz_convert("Europe/Berlin")
regional_features["hour_local"] = regional_features["timestamp_de"].dt.hour
regional_features["day_of_week"] = regional_features["timestamp_de"].dt.dayofweek
regional_features["month"] = regional_features["timestamp_de"].dt.month
regional_features["is_weekend"] = regional_features["day_of_week"] >= 5
regional_features["season"] = regional_features["month"].map({
    12: "winter", 1: "winter", 2: "winter",
    3: "spring", 4: "spring", 5: "spring",
    6: "summer", 7: "summer", 8: "summer",
    9: "autumn", 10: "autumn", 11: "autumn",
})

regional_feature_columns = [
    "wind_speed_100m_avg",
    "wind_direction_100m_sin_avg",
    "wind_direction_100m_cos_avg",
    "solar_radiation_avg",
    "cloud_cover_avg",
    "sunshine_duration_avg",
    "temperature_2m_avg",
]

print(f"Regional weather feature file: {REGIONAL_FEATURES_PATH}")
print(f"Rows: {len(regional_features):,}")
print(f"Regions: {regional_features['region'].nunique()}")
regional_features.head()


### Quick regional weather-feature quality audit before EDA


In [ ]:
coverage_by_region = (
    regional_features.groupby("region")["timestamp_utc"]
    .agg(first_timestamp="min", last_timestamp="max", hourly_rows="size", unique_hours="nunique")
)
quality_summary = pd.Series({
    "rows": len(regional_features),
    "regions": regional_features["region"].nunique(),
    "first_timestamp": regional_features["timestamp_utc"].min(),
    "last_timestamp": regional_features["timestamp_utc"].max(),
    "duplicate_timestamp_region_keys": regional_features.duplicated(["timestamp_utc", "region"]).sum(),
    "total_missing_regional_values": regional_features[regional_feature_columns].isna().sum().sum(),
}, name="value")

missing_by_regional_variable = regional_features[regional_feature_columns].isna().sum().to_frame("missing_rows")
regional_range_audit = pd.DataFrame([
    {
        "variable": variable,
        "observed_min": regional_features[variable].min(),
        "observed_max": regional_features[variable].max(),
    }
    for variable in regional_feature_columns
]).set_index("variable")

display(quality_summary.to_frame())
display(coverage_by_region)
display(missing_by_regional_variable)
display(regional_range_audit)


## Data Inspection and Preprocessing 

### Wind Region Distributions

Use the regional-average table for the main EDA. Reconstruct regional wind direction from its sine/cosine averages and visualize it with wind roses. `direction_resultant_strength` indicates how coherent the point-level directions are within a region; values near zero mean the reconstructed regional direction is less reliable.

In [ ]:
wind_regions = ["offshore_north_sea", "offshore_baltic_sea", "north_wind", "northeast_wind"]
regional_wind = regional_features[regional_features["region"].isin(wind_regions)].copy()

regional_wind["wind_direction_100m_avg"] = (
    np.degrees(np.arctan2(
        regional_wind["wind_direction_100m_sin_avg"],
        regional_wind["wind_direction_100m_cos_avg"],
    )) % 360
)
regional_wind["direction_resultant_strength"] = np.hypot(
    regional_wind["wind_direction_100m_sin_avg"],
    regional_wind["wind_direction_100m_cos_avg"],
)

# Regional-average wind-speed distributions.
fig, ax = plt.subplots(figsize=(11, 5))
ax.boxplot(
    [regional_wind.loc[regional_wind["region"] == region, "wind_speed_100m_avg"] for region in wind_regions],
    tick_labels=wind_regions,
    showfliers=False,
)
ax.set(ylabel="Regional average wind speed at 100m (m/s)", title="Regional wind-speed distributions")
ax.tick_params(axis="x", rotation=20)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# Overlaid distributions make differences in shape and tails easier to compare.
fig, ax = plt.subplots(figsize=(10, 5))
for region in wind_regions:
    values = regional_wind.loc[regional_wind["region"] == region, "wind_speed_100m_avg"]
    ax.hist(values, bins=50, density=True, histtype="step", linewidth=1.8, label=region)
ax.set(xlabel="Regional average wind speed at 100m (m/s)", ylabel="Density", title="Regional wind-speed density")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

# Wind roses: direction frequency stacked by wind-speed band.
direction_edges = np.arange(-11.25, 371.25, 22.5)
direction_centers = np.arange(0, 360, 22.5)
speed_edges = [0, 4, 8, 12, 16, np.inf]
speed_labels = ["0-4", "4-8", "8-12", "12-16", "16+"]

fig, axes = plt.subplots(1, len(wind_regions), figsize=(5 * len(wind_regions), 6), subplot_kw={"projection": "polar"})
axes = np.atleast_1d(axes)
for ax, region in zip(axes, wind_regions):
    region_data = regional_wind[regional_wind["region"] == region].copy()
    wrapped_direction = region_data["wind_direction_100m_avg"].where(
        region_data["wind_direction_100m_avg"] < 348.75,
        region_data["wind_direction_100m_avg"] - 360,
    )
    region_data["direction_bin"] = pd.cut(wrapped_direction, direction_edges, labels=False, right=False)
    region_data["speed_bin"] = pd.cut(region_data["wind_speed_100m_avg"], speed_edges, labels=speed_labels, right=False)
    rose = pd.crosstab(region_data["direction_bin"], region_data["speed_bin"], normalize="all")
    rose = rose.reindex(index=range(16), columns=speed_labels, fill_value=0)

    bottom = np.zeros(16)
    for speed_band in speed_labels:
        frequencies = rose[speed_band].to_numpy() * 100
        ax.bar(np.deg2rad(direction_centers), frequencies, width=np.deg2rad(22.5), bottom=bottom, label=speed_band)
        bottom += frequencies

    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_title(region)
    ax.set_ylabel("Frequency (%)")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, title="Wind speed (m/s)", loc="center right")
fig.suptitle("Regional-average wind roses", y=0.98)
plt.tight_layout(rect=[0, 0, 0.90, 0.96])
plt.show()

regional_wind.groupby("region")[["wind_speed_100m_avg", "direction_resultant_strength"]].describe()


#### Wind Regional Relevance and Difference

Check whether the sampling points grouped into each wind region move together and whether any single point frequently differs from or dominates the regional average. Point relevance is measured against a leave-one-out regional mean.

In [ ]:
wind_point_level = pd.read_csv(
    OBSERVED_COMBINED_PATH,
    usecols=["timestamp_utc", "point_id", "feature_groups", "wind_speed_100m"],
    parse_dates=["timestamp_utc"],
)
wind_point_level["timestamp_utc"] = pd.to_datetime(wind_point_level["timestamp_utc"], utc=True)
wind_point_level["region"] = wind_point_level["feature_groups"].str.split("|")
wind_point_level = wind_point_level.explode("region", ignore_index=True)
wind_point_level = wind_point_level[wind_point_level["region"].str.endswith("wind_points")].copy()
wind_point_level["region"] = wind_point_level["region"].str.removesuffix("_points")
multi_point_wind_regions = ["north_wind", "northeast_wind"]
wind_point_level = wind_point_level[wind_point_level["region"].isin(multi_point_wind_regions)].copy()
wind_point_level = wind_point_level.sort_values(["region", "point_id", "timestamp_utc"])

# Compare each point with the other points in its assigned region.
region_hour_group = wind_point_level.groupby(["timestamp_utc", "region"])["wind_speed_100m"]
wind_point_level["regional_mean"] = region_hour_group.transform("mean")
wind_point_level["leave_one_out_mean"] = (
    (region_hour_group.transform("sum") - wind_point_level["wind_speed_100m"])
    / (region_hour_group.transform("count") - 1)
)
wind_point_level["deviation_from_regional_mean"] = wind_point_level["wind_speed_100m"] - wind_point_level["regional_mean"]
wind_point_level["abs_deviation_from_regional_mean"] = wind_point_level["deviation_from_regional_mean"].abs()
wind_point_level["point_ramp_1h"] = wind_point_level.groupby(["region", "point_id"])["wind_speed_100m"].diff()
wind_point_level["leave_one_out_ramp_1h"] = wind_point_level.groupby(["region", "point_id"])["leave_one_out_mean"].diff()

abs_point_ramp = wind_point_level["point_ramp_1h"].abs()
total_abs_ramp = abs_point_ramp.groupby([wind_point_level["timestamp_utc"], wind_point_level["region"]]).transform("sum")
largest_abs_ramp = abs_point_ramp.groupby([wind_point_level["timestamp_utc"], wind_point_level["region"]]).transform("max")
wind_point_level["abs_ramp_share"] = abs_point_ramp.div(total_abs_ramp.where(total_abs_ramp > 0))
wind_point_level["has_largest_abs_ramp"] = abs_point_ramp.eq(largest_abs_ramp) & total_abs_ramp.gt(0)

point_relevance_rows = []
for (region, point_id), point_data in wind_point_level.groupby(["region", "point_id"]):
    point_relevance_rows.append({
        "region": region,
        "point_id": point_id,
        "point_vs_leave_one_out_corr": point_data["wind_speed_100m"].corr(point_data["leave_one_out_mean"]),
        "ramp_vs_leave_one_out_ramp_corr": point_data["point_ramp_1h"].corr(point_data["leave_one_out_ramp_1h"]),
        "mean_abs_deviation_mps": point_data["abs_deviation_from_regional_mean"].mean(),
        "p95_abs_deviation_mps": point_data["abs_deviation_from_regional_mean"].quantile(0.95),
        "share_abs_deviation_above_2mps": point_data["abs_deviation_from_regional_mean"].gt(2).mean(),
        "mean_abs_ramp_share": point_data["abs_ramp_share"].mean(),
        "share_with_largest_abs_ramp": point_data["has_largest_abs_ramp"].mean(),
    })

wind_point_relevance = pd.DataFrame(point_relevance_rows).set_index(["region", "point_id"])
wind_region_spread = (
    wind_point_level.groupby(["timestamp_utc", "region"])["wind_speed_100m"]
    .agg(regional_mean="mean", regional_std="std", regional_min="min", regional_max="max")
    .reset_index()
)
wind_region_spread["regional_range"] = wind_region_spread["regional_max"] - wind_region_spread["regional_min"]

display(wind_point_relevance.round(3))
display(wind_region_spread.groupby("region")[["regional_std", "regional_range"]].describe().round(3))


In [ ]:
# Pairwise point correlations show whether each multi-point wind region is internally coherent.
fig, axes = plt.subplots(1, len(multi_point_wind_regions), figsize=(6 * len(multi_point_wind_regions), 5), layout="constrained")
axes = np.atleast_1d(axes)
for ax, region in zip(axes, multi_point_wind_regions):
    region_points = wind_point_level[wind_point_level["region"] == region].pivot(
        index="timestamp_utc", columns="point_id", values="wind_speed_100m"
    )
    correlation = region_points.corr()
    image = ax.imshow(correlation, vmin=0, vmax=1, cmap="viridis")
    ax.set_xticks(range(len(correlation)), correlation.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(correlation)), correlation.index)
    ax.set_title(region)
    for row in range(len(correlation)):
        for column in range(len(correlation)):
            ax.text(column, row, f"{correlation.iloc[row, column]:.2f}", ha="center", va="center", color="white")

fig.colorbar(image, ax=axes, label="Pearson correlation", shrink=0.8)
fig.suptitle("Within-region sampling-point wind-speed correlations")
plt.show()

# Regional range summarizes how different the grouped points are in the same hour.
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].boxplot(
    [wind_region_spread.loc[wind_region_spread["region"] == region, "regional_range"] for region in multi_point_wind_regions],
    tick_labels=multi_point_wind_regions,
    showfliers=False,
)
axes[0].set(ylabel="Max minus min wind speed (m/s)", title="Hourly within-region wind-speed range")
axes[0].grid(axis="y", alpha=0.3)

plot_relevance = wind_point_relevance.reset_index()
point_labels = plot_relevance["region"] + ": " + plot_relevance["point_id"]
axes[1].barh(point_labels, plot_relevance["p95_abs_deviation_mps"])
axes[1].set(xlabel="95th percentile absolute deviation (m/s)", title="Sampling-point deviation from regional mean")
axes[1].grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()


It seems the two off shore region should be treated separately.

#### Wind Data Trend and Seasonality

In [ ]:
# Aggregate hourly values to daily means and remove the final one-hour local-date bucket.
daily_wind = (
    regional_wind.set_index("timestamp_de")
    .groupby("region")
    .resample("D")
    .agg(
        wind_speed_100m_avg=("wind_speed_100m_avg", "mean"),
        wind_direction_100m_sin_avg=("wind_direction_100m_sin_avg", "mean"),
        wind_direction_100m_cos_avg=("wind_direction_100m_cos_avg", "mean"),
        hourly_count=("wind_speed_100m_avg", "count"),
    )
    .reset_index()
)
daily_wind = daily_wind[daily_wind["hourly_count"] > 1].copy()

daily_wind["wind_speed_100m_avg_30d"] = (
    daily_wind.groupby("region")["wind_speed_100m_avg"]
    .transform(lambda values: values.rolling(30, center=True, min_periods=7).mean())
)


fig, axes = plt.subplots(len(wind_regions), 1, figsize=(16, 3.8 * len(wind_regions)), sharex=True, sharey=True)
axes = np.atleast_1d(axes)
for ax, region in zip(axes, wind_regions):
    region_data = daily_wind[daily_wind["region"] == region]
    ax.plot(region_data["timestamp_de"], region_data["wind_speed_100m_avg"], alpha=0.18, linewidth=0.7, label="Daily mean")
    ax.plot(region_data["timestamp_de"], region_data["wind_speed_100m_avg_30d"], linewidth=1.8, label="30-day rolling mean")
    ax.set_title(region)
    ax.set_ylabel("Wind speed (m/s)")
    ax.grid(alpha=0.25)
axes[0].legend()
axes[-1].set_xlabel("Germany local date")
fig.suptitle("Regional wind speed over time")
plt.tight_layout()
plt.show()



In [ ]:
# Average regional wind speed by Germany local hour and season across all years.
season_order = ["winter", "spring", "summer", "autumn"]
wind_diurnal = (
    regional_wind.groupby(["region", "season", "hour_local"], as_index=False)["wind_speed_100m_avg"]
    .mean()
)

fig, axes = plt.subplots(1, len(wind_regions), figsize=(5 * len(wind_regions), 5), sharex=True, sharey=True)
axes = np.atleast_1d(axes)
for ax, region in zip(axes, wind_regions):
    region_data = wind_diurnal[wind_diurnal["region"] == region]
    for season in season_order:
        season_data = region_data[region_data["season"] == season]
        ax.plot(
            season_data["hour_local"],
            season_data["wind_speed_100m_avg"],
            marker="o",
            markersize=3,
            linewidth=2,
            label=season,
        )
    ax.set_title(region)
    ax.set_xticks(range(0, 24, 3))
    ax.set_xlabel("Hour of day (Europe/Berlin)")
    ax.grid(alpha=0.3)

axes[0].set_ylabel("Average wind speed at 100m (m/s)")
axes[-1].legend(title="Season")
fig.suptitle("Regional wind-speed diurnal cycle by season")
plt.tight_layout()
plt.show()


In [ ]:
# Circular average wind direction by Germany local hour and season.
wind_direction_diurnal = (
    regional_wind.groupby(["region", "season", "hour_local"], as_index=False)
    .agg(
        direction_sin_avg=("wind_direction_100m_sin_avg", "mean"),
        direction_cos_avg=("wind_direction_100m_cos_avg", "mean"),
    )
)
wind_direction_diurnal["wind_direction_100m_avg"] = (
    np.degrees(np.arctan2(
        wind_direction_diurnal["direction_sin_avg"],
        wind_direction_diurnal["direction_cos_avg"],
    )) % 360
)
wind_direction_diurnal["direction_resultant_strength"] = np.hypot(
    wind_direction_diurnal["direction_sin_avg"],
    wind_direction_diurnal["direction_cos_avg"],
)

fig, axes = plt.subplots(1, len(wind_regions), figsize=(5 * len(wind_regions), 5), sharex=True, sharey=True)
axes = np.atleast_1d(axes)
for ax, region in zip(axes, wind_regions):
    region_data = wind_direction_diurnal[wind_direction_diurnal["region"] == region]
    for season in season_order:
        season_data = region_data[region_data["season"] == season].copy()
        direction = season_data["wind_direction_100m_avg"].copy()
        direction.loc[direction.diff().abs() > 180] = np.nan
        ax.plot(
            season_data["hour_local"],
            direction,
            marker="o",
            markersize=3,
            linewidth=2,
            label=season,
        )
    ax.set_title(region)
    ax.set_xticks(range(0, 24, 3))
    ax.set_xlabel("Hour of day (Europe/Berlin)")
    ax.set_yticks([0, 90, 180, 270, 360], labels=["N 0", "E 90", "S 180", "W 270", "N 360"])
    ax.grid(alpha=0.3)

axes[0].set_ylabel("Circular average wind direction")
axes[-1].legend(title="Season")
fig.suptitle("Regional wind-direction diurnal cycle by season")
plt.tight_layout()
plt.show()

wind_direction_diurnal.groupby(["region", "season"])["direction_resultant_strength"].mean().unstack()


### Load SMARD data for generation and target EDA

The sections above inspect weather data only. SMARD is loaded here, just before the first weather-to-generation join.


In [ ]:
smard_columns = [
    "Start_date",
    "Wind_offshore",
    "Wind_onshore",
    "Photovolt",
    "Renewables_tot",
    "Total_gen",
    "Residuals",
    "Price",
    "Negative_price",
]

SMARD_DATA_PATH = RAW_DATA_PATH / "Combined_smard_data.csv"
smard = pd.read_csv(SMARD_DATA_PATH, usecols=smard_columns, parse_dates=["Start_date"])
print(f"SMARD data file: {SMARD_DATA_PATH}")

# The first repeated local 02:00 is DST (UTC+2); the second is standard time (UTC+1).
first_repeated_hour = smard["Start_date"].duplicated(keep="last")
smard["timestamp_utc"] = (
    smard["Start_date"]
    .dt.tz_localize("Europe/Berlin", ambiguous=first_repeated_hour.to_numpy(), nonexistent="raise")
    .dt.tz_convert("UTC")
)

smard = smard.rename(columns={
    "Wind_offshore": "wind_offshore_mwh",
    "Wind_onshore": "wind_onshore_mwh",
    "Photovolt": "solar_mwh",
    "Renewables_tot": "renewables_total_mwh",
    "Total_gen": "generation_total_mwh",
    "Residuals": "residuals_mwh",
    "Price": "price_eur_per_mwh",
    "Negative_price": "negative_price",
})
smard["wind_total_mwh"] = smard["wind_onshore_mwh"] + smard["wind_offshore_mwh"]
smard["wind_total_fraction"] = smard["wind_total_mwh"] / smard["generation_total_mwh"]
smard["solar_fraction"] = smard["solar_mwh"] / smard["generation_total_mwh"]

smard_features = smard[[
    "timestamp_utc",
    "wind_offshore_mwh",
    "wind_onshore_mwh",
    "wind_total_mwh",
    "solar_mwh",
    "renewables_total_mwh",
    "generation_total_mwh",
    "wind_total_fraction",
    "solar_fraction",
    "residuals_mwh",
    "price_eur_per_mwh",
    "negative_price",
]].sort_values("timestamp_utc").reset_index(drop=True)

if smard_features["timestamp_utc"].duplicated().any():
    raise ValueError("SMARD UTC timestamps are not unique.")


### Hourly Wind Speed vs Generation

Compare hourly regional wind-speed proxies with German onshore and offshore wind generation. The onshore proxy is the simple mean of `north_wind` and `northeast_wind`; the offshore proxy is the mean of `offshore_north_sea` and `offshore_baltic_sea`.

In [ ]:
wind_speed_hourly = regional_wind.pivot(
    index="timestamp_utc",
    columns="region",
    values="wind_speed_100m_avg",
)
wind_speed_hourly["onshore_wind_speed"] = wind_speed_hourly[["north_wind", "northeast_wind"]].mean(axis=1)
wind_speed_hourly["offshore_wind_speed"] = wind_speed_hourly[["offshore_north_sea", "offshore_baltic_sea"]].mean(axis=1)

wind_generation_hourly = (
    wind_speed_hourly[["onshore_wind_speed", "offshore_wind_speed"]]
    .join(
        smard_features.set_index("timestamp_utc")[["wind_onshore_mwh", "wind_offshore_mwh"]],
        how="inner",
    )
    .dropna()
)

print(f"Matched hourly rows: {len(wind_generation_hourly):,}")
print(f"UTC range: {wind_generation_hourly.index.min()} to {wind_generation_hourly.index.max()}")
wind_generation_hourly.head()


In [ ]:
# Hexbin color indicates how many hourly observations occupy each wind-speed/generation area.
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
hexbin_specs = [
    ("onshore_wind_speed", "wind_onshore_mwh", "Onshore"),
    ("offshore_wind_speed", "wind_offshore_mwh", "Offshore"),
]

for ax, (speed_column, generation_column, label) in zip(axes, hexbin_specs):
    hexbin = ax.hexbin(
        wind_generation_hourly[speed_column],
        wind_generation_hourly[generation_column],
        gridsize=55,
        mincnt=1,
        bins="log",
        cmap="viridis",
    )
    ax.set(
        xlabel="Hourly wind-speed proxy at 100m (m/s)",
        ylabel=f"Hourly wind {label.lower()} generation",
        title=f"{label} wind speed vs generation",
    )
    ax.grid(alpha=0.2)
    fig.colorbar(hexbin, ax=ax, label="log10(hour count)")

fig.suptitle("Hourly wind speed and generation density")
plt.tight_layout()
plt.show()


In [ ]:
# Estimate empirical generation curves using shared 0.5 m/s wind-speed bins.
speed_bin_edges = np.arange(0, 31, 0.5)
binned_generation_curves = {}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, (speed_column, generation_column, label) in zip(axes, hexbin_specs):
    analysis_data = wind_generation_hourly[[speed_column, generation_column]].copy()
    analysis_data["speed_bin"] = pd.cut(analysis_data[speed_column], bins=speed_bin_edges)
    curve = (
        analysis_data.groupby("speed_bin", observed=True)
        .agg(
            wind_speed=(speed_column, "mean"),
            generation_mean=(generation_column, "mean"),
            generation_p10=(generation_column, lambda values: values.quantile(0.10)),
            generation_p90=(generation_column, lambda values: values.quantile(0.90)),
            count=(generation_column, "size"),
        )
        .query("count >= 30")
    )
    binned_generation_curves[label.lower()] = curve

    ax.plot(curve["wind_speed"], curve["generation_mean"], linewidth=2, marker="o", markersize=3)
    ax.fill_between(
        curve["wind_speed"],
        curve["generation_p10"],
        curve["generation_p90"],
        alpha=0.2,
        label="10th–90th percentile",
    )
    ax.set(
        xlabel="Hourly wind-speed proxy at 100m (m/s)",
        ylabel=f"Mean hourly wind {label.lower()} generation",
        title=f"{label} empirical generation curve",
    )
    ax.grid(alpha=0.3)
    ax.legend()

fig.suptitle("Binned hourly wind speed and generation")
plt.tight_layout()
plt.show()


In [ ]:
from scipy.stats import spearmanr

correlation_rows = []
for speed_column, generation_column, label in hexbin_specs:
    speed = wind_generation_hourly[speed_column]
    generation = wind_generation_hourly[generation_column]
    correlation_rows.append({
        "generation_type": label.lower(),
        "hourly_rows": len(wind_generation_hourly),
        "pearson_wind_speed": speed.corr(generation),
        "spearman_wind_speed": spearmanr(speed, generation).statistic,
        "pearson_wind_speed_cubed": speed.pow(3).corr(generation), # here we use cubed because of the relation p~v^3, p being production and v being wind speed
    })

wind_generation_correlations = pd.DataFrame(correlation_rows).set_index("generation_type")
wind_generation_correlations.round(3)


#### Incremental Effect of Onshore Wind Direction

Fit four nested ordinary least-squares models to determine whether wind direction improves hourly onshore-generation predictions beyond wind speed.

**Variables**

- Target $y$: hourly German onshore wind generation, `wind_onshore_mwh`.
- Wind-speed predictor $v$: simple mean of the hourly `north_wind` and `northeast_wind` regional-average wind speeds at 100 m.
- Direction predictors $s$ and $c$: mean onshore direction-vector components, calculated by averaging the regional sine and cosine components from `north_wind` and `northeast_wind`. Direction angles are not averaged directly because wind direction is circular.

**Models**

1. Linear speed: $y = \beta_0 + \beta_1 v$.
2. Cubic speed: $y = \beta_0 + \beta_1 v + \beta_2 v^2 + \beta_3 v^3$.
3. Cubic + direction: Model 2 plus $\beta_4 s + \beta_5 c$.
4. Cubic + direction interactions: Model 3 plus $\beta_6(vs) + \beta_7(vc)$, allowing the direction effect to change with wind speed.

**Evaluation**

Train on observations before `2025-01-01` and evaluate on observations from `2025-01-01` onward. Compare test-set $R^2$, MAE, and RMSE. Improvements from Model 2 to Models 3 and 4 measure the incremental predictive value of wind direction after accounting for the nonlinear wind-speed relationship.

In [ ]:
# Average vector components across the two onshore regions; never average direction angles directly.
onshore_direction_hourly = (
    regional_wind[regional_wind["region"].isin(["north_wind", "northeast_wind"])]
    .groupby("timestamp_utc")[["wind_direction_100m_sin_avg", "wind_direction_100m_cos_avg"]]
    .mean()
    .rename(columns={
        "wind_direction_100m_sin_avg": "onshore_direction_sin",
        "wind_direction_100m_cos_avg": "onshore_direction_cos",
    })
)

onshore_model_data = (
    wind_generation_hourly[["onshore_wind_speed", "wind_onshore_mwh"]]
    .join(onshore_direction_hourly, how="inner")
    .dropna()
    .sort_index()
)

onshore_model_data["wind_speed_squared"] = onshore_model_data["onshore_wind_speed"].pow(2)
onshore_model_data["wind_speed_cubed"] = onshore_model_data["onshore_wind_speed"].pow(3)
onshore_model_data["wind_speed_x_direction_sin"] = (
    onshore_model_data["onshore_wind_speed"] * onshore_model_data["onshore_direction_sin"]
)
onshore_model_data["wind_speed_x_direction_cos"] = (
    onshore_model_data["onshore_wind_speed"] * onshore_model_data["onshore_direction_cos"]
)

onshore_model_features = {
    "1. Linear speed": ["onshore_wind_speed"],
    "2. Cubic speed": ["onshore_wind_speed", "wind_speed_squared", "wind_speed_cubed"],
    "3. Cubic + direction": [
        "onshore_wind_speed", "wind_speed_squared", "wind_speed_cubed",
        "onshore_direction_sin", "onshore_direction_cos",
    ],
    "4. Cubic + direction interactions": [
        "onshore_wind_speed", "wind_speed_squared", "wind_speed_cubed",
        "onshore_direction_sin", "onshore_direction_cos",
        "wind_speed_x_direction_sin", "wind_speed_x_direction_cos",
    ],
}

train_mask = onshore_model_data.index < "2025-01-01"
test_mask = ~train_mask
target = onshore_model_data["wind_onshore_mwh"].to_numpy()
onshore_model_results = []
onshore_model_predictions = {}

for model_name, feature_columns in onshore_model_features.items():
    features = onshore_model_data[feature_columns].to_numpy()
    design_matrix = np.column_stack([np.ones(len(features)), features])
    coefficients = np.linalg.lstsq(design_matrix[train_mask], target[train_mask], rcond=None)[0]
    train_predictions = design_matrix[train_mask] @ coefficients
    predictions = design_matrix[test_mask] @ coefficients
    train_actual = target[train_mask]
    actual = target[test_mask]

    train_rmse = np.sqrt(np.mean((train_actual - train_predictions) ** 2))
    mae = np.mean(np.abs(actual - predictions))
    rmse = np.sqrt(np.mean((actual - predictions) ** 2))
    r_squared = 1 - np.sum((actual - predictions) ** 2) / np.sum((actual - actual.mean()) ** 2)
    onshore_model_results.append({
        "model": model_name,
        "features": len(feature_columns),
        "train_rmse": train_rmse,
        "test_r_squared": r_squared,
        "test_mae": mae,
        "test_rmse": rmse,
    })
    onshore_model_predictions[model_name] = predictions

onshore_model_results = pd.DataFrame(onshore_model_results).set_index("model")
print(f"Train rows: {train_mask.sum():,}; test rows: {test_mask.sum():,}")
onshore_model_results.round(3)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metric_specs = [
    ("test_r_squared", "Test R²", "higher is better"),
    ("test_mae", "Test MAE", "lower is better"),
    ("test_rmse", "Test RMSE", "lower is better"),
]

for ax, (metric, title, subtitle) in zip(axes, metric_specs):
    values = onshore_model_results[metric]
    bars = ax.bar(range(len(values)), values)
    ax.set_xticks(range(len(values)), [str(index).split(". ", 1)[0] for index in values.index])
    ax.set_xlabel("Model number")
    ax.set_title(f"{title}\n({subtitle})")
    ax.grid(axis="y", alpha=0.3)
    ax.bar_label(bars, fmt="%.3f" if metric == "test_r_squared" else "%.0f", padding=3)

fig.suptitle("Onshore wind-direction incremental model comparison")
plt.tight_layout()
plt.show()

onshore_model_results.assign(
    delta_r_squared=onshore_model_results["test_r_squared"].diff(),
    delta_mae=onshore_model_results["test_mae"].diff(),
    delta_rmse=onshore_model_results["test_rmse"].diff(),
).round(3)


#### North Sea and Baltic Sea Offshore Proxies

Compare the two original offshore sampling-point wind speeds separately against German offshore wind generation.

In [ ]:
offshore_point_ids = ["offshore_north_sea", "offshore_baltic_sea"]
offshore_point_wind = pd.read_csv(
    OBSERVED_COMBINED_PATH,
    usecols=["timestamp_utc", "point_id", "wind_speed_100m"],
    parse_dates=["timestamp_utc"],
)
offshore_point_wind = offshore_point_wind[offshore_point_wind["point_id"].isin(offshore_point_ids)]
offshore_point_wind = offshore_point_wind.pivot(
    index="timestamp_utc",
    columns="point_id",
    values="wind_speed_100m",
)

offshore_point_generation_hourly = (
    offshore_point_wind
    .join(smard_features.set_index("timestamp_utc")[["wind_offshore_mwh"]], how="inner")
    .dropna()
)

print(f"Matched hourly rows: {len(offshore_point_generation_hourly):,}")
offshore_point_generation_hourly.head()


In [ ]:
offshore_point_specs = [
    ("offshore_north_sea", "North Sea"),
    ("offshore_baltic_sea", "Baltic Sea"),
]
offshore_point_binned_curves = {}

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
for ax, (speed_column, label) in zip(axes, offshore_point_specs):
    analysis_data = offshore_point_generation_hourly[[speed_column, "wind_offshore_mwh"]].copy()
    analysis_data["speed_bin"] = pd.cut(analysis_data[speed_column], bins=speed_bin_edges)
    curve = (
        analysis_data.groupby("speed_bin", observed=True)
        .agg(
            wind_speed=(speed_column, "mean"),
            generation_mean=("wind_offshore_mwh", "mean"),
            generation_p10=("wind_offshore_mwh", lambda values: values.quantile(0.10)),
            generation_p90=("wind_offshore_mwh", lambda values: values.quantile(0.90)),
            count=("wind_offshore_mwh", "size"),
        )
        .query("count >= 30")
    )
    offshore_point_binned_curves[speed_column] = curve

    ax.plot(curve["wind_speed"], curve["generation_mean"], linewidth=2, marker="o", markersize=3)
    ax.fill_between(
        curve["wind_speed"],
        curve["generation_p10"],
        curve["generation_p90"],
        alpha=0.2,
        label="10th–90th percentile",
    )
    ax.set(
        xlabel="Hourly wind speed at 100m (m/s)",
        title=f"{label} proxy vs German offshore generation",
    )
    ax.grid(alpha=0.3)
    ax.legend()

axes[0].set_ylabel("Mean hourly German offshore wind generation")
fig.suptitle("Binned offshore sampling-point wind speed and generation")
plt.tight_layout()
plt.show()


In [ ]:
offshore_point_correlation_rows = []
for speed_column, label in offshore_point_specs:
    speed = offshore_point_generation_hourly[speed_column]
    generation = offshore_point_generation_hourly["wind_offshore_mwh"]
    offshore_point_correlation_rows.append({
        "offshore_proxy": label,
        "hourly_rows": len(offshore_point_generation_hourly),
        "pearson_wind_speed": speed.corr(generation),
        "spearman_wind_speed": spearmanr(speed, generation).statistic,
        "pearson_wind_speed_cubed": speed.pow(3).corr(generation),
    })

offshore_point_correlations = pd.DataFrame(offshore_point_correlation_rows).set_index("offshore_proxy")
offshore_point_correlations.round(3)


### Solar Region Radiation Patterns

Compare daily mean shortwave radiation and the strong seasonal diurnal cycle across the two primary solar-generation regions.

In [ ]:
solar_regions = ["south_solar", "east_solar"]
regional_solar = regional_features[regional_features["region"].isin(solar_regions)].copy()

# Aggregate hourly radiation to daily means using Germany local dates.
daily_solar = (
    regional_solar.set_index("timestamp_de")
    .groupby("region")
    .resample("D")
    .agg(
        solar_radiation_avg=("solar_radiation_avg", "mean"),
        hourly_count=("solar_radiation_avg", "count"),
    )
    .reset_index()
)
daily_solar = daily_solar[daily_solar["hourly_count"] > 1].copy()
daily_solar["solar_radiation_avg_30d"] = (
    daily_solar.groupby("region")["solar_radiation_avg"]
    .transform(lambda values: values.rolling(30, center=True, min_periods=7).mean())
)

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True, sharey=True)
for ax, region in zip(axes, solar_regions):
    region_data = daily_solar[daily_solar["region"] == region]
    ax.plot(
        region_data["timestamp_de"],
        region_data["solar_radiation_avg"],
        alpha=0.25,
        linewidth=0.7,
        label="Daily mean",
    )
    ax.plot(
        region_data["timestamp_de"],
        region_data["solar_radiation_avg_30d"],
        linewidth=2,
        label="30-day centered mean",
    )
    ax.set_title(region)
    ax.set_ylabel("Shortwave radiation (W/m²)")
    ax.grid(alpha=0.3)
    ax.legend()

axes[-1].set_xlabel("Date (Europe/Berlin)")
fig.suptitle("Regional daily mean solar radiation")
plt.tight_layout()
plt.show()


In [ ]:
# Exclude nighttime zero-radiation hours so they do not dominate the comparison.
daylight_solar = regional_solar[regional_solar["solar_radiation_avg"] > 0].copy()

fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot(
    [daylight_solar.loc[daylight_solar["region"] == region, "solar_radiation_avg"] for region in solar_regions],
    tick_labels=solar_regions,
    showfliers=False,
)
ax.set(
    ylabel="Shortwave radiation (W/m²)",
    title="Daylight-only regional solar-radiation distributions",
)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# Use shared histogram bins and density lines so similar regional distributions remain visible.
fig, ax = plt.subplots(figsize=(10, 5))
bin_edges = np.linspace(0, daylight_solar["solar_radiation_avg"].max(), 61)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
for region in solar_regions:
    values = daylight_solar.loc[daylight_solar["region"] == region, "solar_radiation_avg"]
    density, _ = np.histogram(values, bins=bin_edges, density=True)
    ax.plot(bin_centers, density, linewidth=2, label=region)
ax.set(
    xlabel="Shortwave radiation (W/m²)",
    ylabel="Density",
    title="Daylight-only regional solar-radiation density curves",
)
ax.grid(axis="y", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

daylight_solar.groupby("region")["solar_radiation_avg"].describe()


In [ ]:
# Average regional solar radiation by Germany local hour and season across all years.
solar_diurnal = (
    regional_solar.groupby(["region", "season", "hour_local"], as_index=False)["solar_radiation_avg"]
    .mean()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=True)
for ax, region in zip(axes, solar_regions):
    region_data = solar_diurnal[solar_diurnal["region"] == region]
    for season in season_order:
        season_data = region_data[region_data["season"] == season]
        ax.plot(
            season_data["hour_local"],
            season_data["solar_radiation_avg"],
            marker="o",
            markersize=3,
            linewidth=2,
            label=season,
        )
    ax.set_title(region)
    ax.set_xticks(range(0, 24, 3))
    ax.set_xlabel("Hour of day (Europe/Berlin)")
    ax.grid(alpha=0.3)

axes[0].set_ylabel("Average shortwave radiation (W/m²)")
axes[-1].legend(title="Season")
fig.suptitle("Regional solar-radiation diurnal cycle by season")
plt.tight_layout()
plt.show()


#### Solar Regional Relevance and Difference

Check whether the sampling points grouped into each solar region move together and whether any single point frequently differs from or dominates the regional average. Correlation and deviation metrics use daylight hours only because shared nighttime zeros would artificially inflate agreement. Point relevance is measured against a leave-one-out regional mean.

In [ ]:
solar_point_level = pd.read_csv(
    OBSERVED_COMBINED_PATH,
    usecols=["timestamp_utc", "point_id", "feature_groups", "shortwave_radiation"],
    parse_dates=["timestamp_utc"],
)
solar_point_level["timestamp_utc"] = pd.to_datetime(solar_point_level["timestamp_utc"], utc=True)
solar_point_level["region"] = solar_point_level["feature_groups"].str.split("|")
solar_point_level = solar_point_level.explode("region", ignore_index=True)
solar_point_level = solar_point_level[solar_point_level["region"].str.endswith("solar_points")].copy()
solar_point_level["region"] = solar_point_level["region"].str.removesuffix("_points")
solar_point_level = solar_point_level.sort_values(["region", "point_id", "timestamp_utc"])

# Compare each point with the other points in its assigned region.
region_hour_group = solar_point_level.groupby(["timestamp_utc", "region"])["shortwave_radiation"]
solar_point_level["regional_mean"] = region_hour_group.transform("mean")
solar_point_level["leave_one_out_mean"] = (
    (region_hour_group.transform("sum") - solar_point_level["shortwave_radiation"])
    / (region_hour_group.transform("count") - 1)
)
solar_point_level["deviation_from_regional_mean"] = solar_point_level["shortwave_radiation"] - solar_point_level["regional_mean"]
solar_point_level["abs_deviation_from_regional_mean"] = solar_point_level["deviation_from_regional_mean"].abs()
solar_point_level["point_ramp_1h"] = solar_point_level.groupby(["region", "point_id"])["shortwave_radiation"].diff()
solar_point_level["leave_one_out_ramp_1h"] = solar_point_level.groupby(["region", "point_id"])["leave_one_out_mean"].diff()

abs_point_ramp = solar_point_level["point_ramp_1h"].abs()
total_abs_ramp = abs_point_ramp.groupby([solar_point_level["timestamp_utc"], solar_point_level["region"]]).transform("sum")
largest_abs_ramp = abs_point_ramp.groupby([solar_point_level["timestamp_utc"], solar_point_level["region"]]).transform("max")
solar_point_level["abs_ramp_share"] = abs_point_ramp.div(total_abs_ramp.where(total_abs_ramp > 0))
solar_point_level["has_largest_abs_ramp"] = abs_point_ramp.eq(largest_abs_ramp) & total_abs_ramp.gt(0)

daylight_solar_points = solar_point_level[solar_point_level["regional_mean"] > 0].copy()
point_relevance_rows = []
for (region, point_id), point_data in daylight_solar_points.groupby(["region", "point_id"]):
    point_relevance_rows.append({
        "region": region,
        "point_id": point_id,
        "point_vs_leave_one_out_corr": point_data["shortwave_radiation"].corr(point_data["leave_one_out_mean"]),
        "ramp_vs_leave_one_out_ramp_corr": point_data["point_ramp_1h"].corr(point_data["leave_one_out_ramp_1h"]),
        "mean_abs_deviation_wm2": point_data["abs_deviation_from_regional_mean"].mean(),
        "p95_abs_deviation_wm2": point_data["abs_deviation_from_regional_mean"].quantile(0.95),
        "share_abs_deviation_above_100wm2": point_data["abs_deviation_from_regional_mean"].gt(100).mean(),
        "mean_abs_ramp_share": point_data["abs_ramp_share"].mean(),
        "share_with_largest_abs_ramp": point_data["has_largest_abs_ramp"].mean(),
    })

solar_point_relevance = pd.DataFrame(point_relevance_rows).set_index(["region", "point_id"])
solar_region_spread = (
    daylight_solar_points.groupby(["timestamp_utc", "region"])["shortwave_radiation"]
    .agg(regional_mean="mean", regional_std="std", regional_min="min", regional_max="max")
    .reset_index()
)
solar_region_spread["regional_range"] = solar_region_spread["regional_max"] - solar_region_spread["regional_min"]

display(solar_point_relevance.round(3))
display(solar_region_spread.groupby("region")[["regional_std", "regional_range"]].describe().round(3))


In [ ]:
# Daylight-only pairwise correlations show whether each proposed solar region is internally coherent.
fig, axes = plt.subplots(1, 2, figsize=(14, 5), layout="constrained")
for ax, region in zip(axes, solar_regions):
    region_points = daylight_solar_points[daylight_solar_points["region"] == region].pivot(
        index="timestamp_utc", columns="point_id", values="shortwave_radiation"
    )
    correlation = region_points.corr()
    image = ax.imshow(correlation, vmin=0, vmax=1, cmap="viridis")
    ax.set_xticks(range(len(correlation)), correlation.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(correlation)), correlation.index)
    ax.set_title(region)
    for row in range(len(correlation)):
        for column in range(len(correlation)):
            ax.text(column, row, f"{correlation.iloc[row, column]:.2f}", ha="center", va="center", color="white")

fig.colorbar(image, ax=axes, label="Pearson correlation", shrink=0.8)
fig.suptitle("Within-region sampling-point daylight radiation correlations")
plt.show()

# Regional range summarizes how different the grouped points are in the same daylight hour.
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].boxplot(
    [solar_region_spread.loc[solar_region_spread["region"] == region, "regional_range"] for region in solar_regions],
    tick_labels=solar_regions,
    showfliers=False,
)
axes[0].set(ylabel="Max minus min radiation (W/m²)", title="Daylight within-region radiation range")
axes[0].grid(axis="y", alpha=0.3)

plot_relevance = solar_point_relevance.reset_index()
point_labels = plot_relevance["region"] + ": " + plot_relevance["point_id"]
axes[1].barh(point_labels, plot_relevance["p95_abs_deviation_wm2"])
axes[1].set(xlabel="95th percentile absolute deviation (W/m²)", title="Sampling-point deviation from regional mean")
axes[1].grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

### Cloud Cover vs Solar Radiation

Measure how cloud cover relates to regional shortwave radiation after excluding true nighttime hours. Potential-daylight hours are identified from the historical 95th-percentile radiation for each region, month, and local hour, which retains fully cloudy daylight observations with zero radiation. Anomalies remove the normal `region × month × local hour` pattern before correlation.

In [ ]:
from scipy.stats import spearmanr

cloud_radiation = regional_solar.copy()
potential_radiation = (
    cloud_radiation.groupby(["region", "month", "hour_local"])["solar_radiation_avg"]
    .transform(lambda values: values.quantile(0.95))
)
cloud_radiation = cloud_radiation[potential_radiation > 0].copy()

# Remove the normal seasonal and diurnal level from both variables.
for variable in ["solar_radiation_avg", "cloud_cover_avg"]:
    expected = cloud_radiation.groupby(["region", "month", "hour_local"])[variable].transform("mean")
    cloud_radiation[f"{variable}_anomaly"] = cloud_radiation[variable] - expected

cloud_radiation_correlation_rows = []
for region, region_data in cloud_radiation.groupby("region"):
    cloud_radiation_correlation_rows.append({
        "region": region,
        "potential_daylight_rows": len(region_data),
        "raw_pearson": region_data["cloud_cover_avg"].corr(region_data["solar_radiation_avg"]),
        "raw_spearman": spearmanr(region_data["cloud_cover_avg"], region_data["solar_radiation_avg"]).statistic,
        "anomaly_pearson": region_data["cloud_cover_avg_anomaly"].corr(region_data["solar_radiation_avg_anomaly"]),
        "anomaly_spearman": spearmanr(region_data["cloud_cover_avg_anomaly"], region_data["solar_radiation_avg_anomaly"]).statistic,
    })

cloud_radiation_correlations = pd.DataFrame(cloud_radiation_correlation_rows).set_index("region")
seasonal_cloud_radiation_correlations = (
    cloud_radiation.groupby(["region", "season"])
    .apply(
        lambda group: pd.Series({
            "raw_pearson": group["cloud_cover_avg"].corr(group["solar_radiation_avg"]),
            "anomaly_pearson": group["cloud_cover_avg_anomaly"].corr(group["solar_radiation_avg_anomaly"]),
        }),
        include_groups=False,
    )
)

cloud_bin_edges = [-0.01, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100.01]
cloud_radiation["cloud_cover_bin"] = pd.cut(cloud_radiation["cloud_cover_avg"], bins=cloud_bin_edges)
cloud_radiation_by_bin = (
    cloud_radiation.groupby(["region", "cloud_cover_bin"], observed=True)
    .agg(
        cloud_cover_mean=("cloud_cover_avg", "mean"),
        radiation_mean=("solar_radiation_avg", "mean"),
        radiation_anomaly_mean=("solar_radiation_avg_anomaly", "mean"),
        count=("solar_radiation_avg", "size"),
    )
    .reset_index()
)

display(cloud_radiation_correlations.round(3))
display(seasonal_cloud_radiation_correlations.round(3))
display(cloud_radiation_by_bin.round(2))

In [ ]:
# Cloud-cover bins provide an interpretable view of the average radiation reduction.
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for region in solar_regions:
    region_data = cloud_radiation_by_bin[cloud_radiation_by_bin["region"] == region]
    axes[0].plot(region_data["cloud_cover_mean"], region_data["radiation_mean"], marker="o", label=region)
    axes[1].plot(region_data["cloud_cover_mean"], region_data["radiation_anomaly_mean"], marker="o", label=region)

axes[0].set(xlabel="Regional average cloud cover (%)", ylabel="Mean shortwave radiation (W/m²)", title="Raw radiation by cloud-cover bin")
axes[1].set(xlabel="Regional average cloud cover (%)", ylabel="Mean radiation anomaly (W/m²)", title="Radiation anomaly by cloud-cover bin")
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()

### Core Weather Variable Redundancy

Compare selected core-variable pairs that may contain overlapping weather information. Solar relationships use potential-daylight hours and compare raw values with anomalies after removing the normal `region × month × local hour` pattern. Wind direction is circular, so its relationship with wind speed is evaluated using direction bins rather than ordinary correlation.

In [ ]:
solar_redundancy_regions = ["south_solar", "east_solar"]
solar_redundancy = cloud_radiation[cloud_radiation["region"].isin(solar_redundancy_regions)].copy()
solar_redundancy_variables = [
    "solar_radiation_avg",
    "sunshine_duration_avg",
    "cloud_cover_avg",
    "temperature_2m_avg",
]
for variable in solar_redundancy_variables:
    anomaly_column = f"{variable}_anomaly"
    if anomaly_column not in solar_redundancy:
        expected = solar_redundancy.groupby(["region", "month", "hour_local"])[variable].transform("mean")
        solar_redundancy[anomaly_column] = solar_redundancy[variable] - expected

solar_pair_specs = [
    ("solar_radiation_avg", "sunshine_duration_avg", "radiation vs sunshine duration"),
    ("solar_radiation_avg", "cloud_cover_avg", "radiation vs cloud cover"),
    ("sunshine_duration_avg", "cloud_cover_avg", "sunshine duration vs cloud cover"),
    ("temperature_2m_avg", "solar_radiation_avg", "temperature vs radiation"),
]

solar_redundancy_rows = []
for region, region_data in solar_redundancy.groupby("region"):
    for first_variable, second_variable, pair_label in solar_pair_specs:
        solar_redundancy_rows.append({
            "region": region,
            "pair": pair_label,
            "raw_pearson": region_data[first_variable].corr(region_data[second_variable]),
            "raw_spearman": spearmanr(region_data[first_variable], region_data[second_variable]).statistic,
            "anomaly_pearson": region_data[f"{first_variable}_anomaly"].corr(region_data[f"{second_variable}_anomaly"]),
            "anomaly_spearman": spearmanr(region_data[f"{first_variable}_anomaly"], region_data[f"{second_variable}_anomaly"]).statistic,
        })

solar_redundancy_correlations = pd.DataFrame(solar_redundancy_rows).set_index(["region", "pair"])
display(solar_redundancy_correlations.round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
for ax, region in zip(axes, solar_redundancy_regions):
    region_results = solar_redundancy_correlations.loc[region].reset_index()
    positions = np.arange(len(region_results))
    width = 0.36
    ax.bar(positions - width / 2, region_results["raw_pearson"], width, label="Raw Pearson")
    ax.bar(positions + width / 2, region_results["anomaly_pearson"], width, label="Anomaly Pearson")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xticks(positions, region_results["pair"], rotation=35, ha="right")
    ax.set_title(region)
    ax.grid(axis="y", alpha=0.3)

axes[0].set_ylabel("Pearson correlation")
axes[0].legend()
fig.suptitle("Selected solar-weather pair correlations")
plt.tight_layout()
plt.show()

In [ ]:
wind_redundancy = regional_wind.copy()
for variable in ["wind_speed_100m_avg", "cloud_cover_avg"]:
    expected = wind_redundancy.groupby(["region", "month", "hour_local"])[variable].transform("mean")
    wind_redundancy[f"{variable}_anomaly"] = wind_redundancy[variable] - expected

wind_speed_cloud_rows = []
for region, region_data in wind_redundancy.groupby("region"):
    wind_speed_cloud_rows.append({
        "region": region,
        "raw_pearson": region_data["wind_speed_100m_avg"].corr(region_data["cloud_cover_avg"]),
        "raw_spearman": spearmanr(region_data["wind_speed_100m_avg"], region_data["cloud_cover_avg"]).statistic,
        "anomaly_pearson": region_data["wind_speed_100m_avg_anomaly"].corr(region_data["cloud_cover_avg_anomaly"]),
        "anomaly_spearman": spearmanr(region_data["wind_speed_100m_avg_anomaly"], region_data["cloud_cover_avg_anomaly"]).statistic,
    })
wind_speed_cloud_correlations = pd.DataFrame(wind_speed_cloud_rows).set_index("region")

direction_labels = ["N", "NNE", "NE", "ENE", "E", "ESE", "SE", "SSE", "S", "SSW", "SW", "WSW", "W", "WNW", "NW", "NNW"]
wind_redundancy["direction_bin_number"] = (((wind_redundancy["wind_direction_100m_avg"] + 11.25) // 22.5) % 16).astype(int)
wind_direction_speed = (
    wind_redundancy.groupby(["region", "direction_bin_number"], as_index=False)
    .agg(mean_wind_speed=("wind_speed_100m_avg", "mean"), median_wind_speed=("wind_speed_100m_avg", "median"), count=("wind_speed_100m_avg", "size"))
)
wind_direction_speed["direction"] = wind_direction_speed["direction_bin_number"].map(dict(enumerate(direction_labels)))

wind_redundancy["wind_speed_bin"] = pd.qcut(wind_redundancy["wind_speed_100m_avg"], q=10, duplicates="drop")
wind_speed_cloud_by_bin = (
    wind_redundancy.groupby(["region", "wind_speed_bin"], observed=True)
    .agg(
        wind_speed_mean=("wind_speed_100m_avg", "mean"),
        cloud_cover_mean=("cloud_cover_avg", "mean"),
        cloud_cover_anomaly_mean=("cloud_cover_avg_anomaly", "mean"),
        count=("cloud_cover_avg", "size"),
    )
    .reset_index()
)

display(wind_speed_cloud_correlations.round(3))
display(wind_direction_speed.pivot(index="region", columns="direction", values="mean_wind_speed").reindex(columns=direction_labels).round(2))

fig, axes = plt.subplots(1, 2, figsize=(17, 5))
for region in wind_regions:
    direction_data = wind_direction_speed[wind_direction_speed["region"] == region].sort_values("direction_bin_number")
    axes[0].plot(direction_data["direction_bin_number"], direction_data["mean_wind_speed"], marker="o", label=region)
    speed_data = wind_speed_cloud_by_bin[wind_speed_cloud_by_bin["region"] == region]
    axes[1].plot(speed_data["wind_speed_mean"], speed_data["cloud_cover_anomaly_mean"], marker="o", label=region)

axes[0].set_xticks(range(16), direction_labels, rotation=45)
axes[0].set(xlabel="Wind direction", ylabel="Mean wind speed at 100m (m/s)", title="Wind speed by direction bin")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set(xlabel="Mean wind speed at 100m (m/s)", ylabel="Mean cloud-cover anomaly (percentage points)", title="Cloud-cover anomaly by wind-speed bin")
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()

### Weather feature hourly ACF

Measure how quickly regional weather information decays over hourly lags. Raw wind speed represents persistent wind conditions. Radiation and cloud-cover anomalies remove the normal `region ? month ? local hour` pattern before ACF calculation.


In [ ]:
max_acf_lag = 168
selected_acf_lags = [1, 3, 6, 12, 24, 48, 72, 120, 168]


def hourly_acf(series, max_lag=max_acf_lag):
    return pd.Series(
        [series.corr(series.shift(lag)) for lag in range(max_lag + 1)],
        index=range(max_lag + 1),
    )


def add_region_mean(frame, column_name="regional_mean"):
    result = frame.copy()
    result[column_name] = result.mean(axis=1)
    return result


# Wind-speed levels by region and across-region mean.
wind_speed_acf_input = add_region_mean(
    regional_wind.pivot(index="timestamp_utc", columns="region", values="wind_speed_100m_avg")[wind_regions]
)

# Standardize against each region-month-hour climatology. Radiation groups with zero
# nighttime variance remain missing so they do not create artificial daily persistence.
solar_acf_data = regional_solar.copy()
for variable in ["solar_radiation_avg", "cloud_cover_avg"]:
    climatology_group = solar_acf_data.groupby(["region", "month", "hour_local"])[variable]
    expected = climatology_group.transform("mean")
    expected_std = climatology_group.transform("std").replace(0, np.nan)
    solar_acf_data[f"{variable}_standardized_anomaly"] = (solar_acf_data[variable] - expected) / expected_std

radiation_anomaly_acf_input = add_region_mean(
    solar_acf_data.pivot(index="timestamp_utc", columns="region", values="solar_radiation_avg_standardized_anomaly")[solar_regions]
)
cloud_anomaly_acf_input = add_region_mean(
    solar_acf_data.pivot(index="timestamp_utc", columns="region", values="cloud_cover_avg_standardized_anomaly")[solar_regions]
)

acf_input_groups = {
    "wind_speed": wind_speed_acf_input,
    "radiation_anomaly": radiation_anomaly_acf_input,
    "cloud_cover_anomaly": cloud_anomaly_acf_input,
}
weather_acf = {
    group_name: pd.DataFrame({column: hourly_acf(group_data[column]) for column in group_data})
    for group_name, group_data in acf_input_groups.items()
}

selected_acf_rows = []
for group_name, group_acf in weather_acf.items():
    for feature in group_acf.columns:
        selected_acf_rows.append({
            "group": group_name,
            "feature": feature,
            **{f"acf_{lag}h": group_acf.loc[lag, feature] for lag in selected_acf_lags},
        })

selected_weather_acf = pd.DataFrame(selected_acf_rows).set_index(["group", "feature"])
display(selected_weather_acf.round(3))

fig, axes = plt.subplots(1, 2, figsize=(18, 5), sharex=True, sharey=True)
acf_plot_specs = [
    ("wind_speed", "Wind-speed ACF"),
    ("radiation_anomaly", "Radiation-anomaly ACF"),
]
for ax, (group_name, title) in zip(axes, acf_plot_specs):
    group_acf = weather_acf[group_name]
    for feature in group_acf.columns:
        ax.plot(group_acf.index, group_acf[feature], linewidth=1.8, label=feature)
    for lag in [24, 48, 72, 168]:
        ax.axvline(lag, color="gray", linewidth=0.8, linestyle="--", alpha=0.5)
    for level in [0.2, 0.5, 0.8]:
        ax.axhline(level, color="gray", linewidth=0.7, linestyle=":", alpha=0.4)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set(title=title, xlabel="Hourly lag", ylabel="Autocorrelation")
    ax.grid(alpha=0.25)
    ax.legend()

fig.suptitle("Hourly weather persistence over 0-168 hour lags")
plt.tight_layout()
plt.show()
